# Structured engineering SVG pilot
Parameter generation, precise edits, deterministic SVG and frame FEM.
600 training / 100 validation / 100 test examples grouped by source design; evaluate 12 examples per task type before and after training. Synthetic portal-only pilot, not an Astra result.
Artifacts: `/content/engsvg-structured-run-artifacts.zip`. Download before runtime expiry.

In [2]:
import pathlib, subprocess, sys
def run(args, **kwargs):
    print('RUN:', ' '.join(args[:4]), flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end='', flush=True)
    if p.wait(): raise RuntimeError('Command failed: '+str(p.returncode))
root = pathlib.Path('/content/SVG')
if not root.exists():
    run(['git','clone','--depth','1','--branch','cvpr2027-research-package','https://github.com/ayushdebnath012/SVG.git',str(root)],check=True)
run([sys.executable,'-m','pip','install','-q','svgpathtools==1.7.2','peft','transformers','accelerate'],check=True)
run([sys.executable,'-m','pip','uninstall','-y','-q','torchao'],check=True)
(root/'cvpr2027/scripts/structured_engsvg.py').write_text('"""Bounded portal-frame pilot: model interprets requests; deterministic code draws/solves.\nNot detached-SVG understanding or whole-table safety certification.\n"""\nimport argparse, copy, hashlib, json, math, random, re\nfrom pathlib import Path\nimport cad_astra_benchmark as B\n\nFIELDS = (\'width_mm\',\'height_mm\',\'section_b_mm\',\'section_h_mm\',\'E_mpa\',\'vertical_load_N\',\'horizontal_load_N\')\nSCHEMA = \'Return JSON only: {"action":"create","parameters":{...}} or {"action":"edit","changes":{...}} or {"action":"clarify","missing":[...]}. Fields: \'+\', \'.join(FIELDS)+\'. All fields are required for creation. Edits change only requested fields. Loads are positive magnitudes: downward vertical and rightward horizontal at top-right. This is a planar rigid-joint portal with fixed bases, uniform rectangular section, no self-weight. Never invent missing values. \'\n\ndef digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True).encode()).hexdigest()\ndef validate(p):\n    if set(p)!=set(FIELDS): raise ValueError(\'missing or extra parameters\')\n    for k,v in p.items():\n        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v): raise ValueError(\'nonfinite/nonnumeric parameter\')\n        if v < 0 or (k not in FIELDS[-2:] and v<=0): raise ValueError(\'invalid parameter range\')\n    if p[\'width_mm\']>20000 or p[\'height_mm\']>20000: raise ValueError(\'outside pilot envelope\')\n\ndef execute(source, action):\n    kind=action.get(\'action\')\n    if kind==\'clarify\':\n        if set(action)!={\'action\',\'missing\'} or not isinstance(action[\'missing\'],list) or not action[\'missing\'] or any(k not in FIELDS for k in action[\'missing\']): raise ValueError(\'invalid clarification\')\n        return None\n    if kind==\'create\' and source is None and set(action)=={\'action\',\'parameters\'}: p=copy.deepcopy(action[\'parameters\'])\n    elif kind==\'edit\' and source is not None and set(action)=={\'action\',\'changes\'}:\n        if not isinstance(action[\'changes\'],dict) or not action[\'changes\'] or not set(action[\'changes\'])<=set(FIELDS): raise ValueError(\'invalid changes\')\n        p=copy.deepcopy(source); p.update(action[\'changes\'])\n    else: raise ValueError(\'invalid action\')\n    validate(p)\n    return p\n\ndef model(p):\n    validate(p)\n    w,h=p[\'width_mm\'],p[\'height_mm\']\n    return dict(name=\'Table side-frame: fixed bases, rigid joints\',nodes={\'A\':[0,0],\'B\':[w,0],\'C\':[0,h],\'D\':[w,h]},\n      members=[B.member(i,a,b,p[\'section_b_mm\'],p[\'section_h_mm\'],p[\'E_mpa\']) for i,a,b in [(\'left\',\'A\',\'C\'),(\'right\',\'B\',\'D\'),(\'top\',\'C\',\'D\')]],\n      supports={\'A\':[0,1,2],\'B\':[0,1,2]},loads={\'D\':[p[\'horizontal_load_N\'],-p[\'vertical_load_N\'],0]},probe=\'D\')\n\ndef artifact(p):\n    m=model(p); r=B.solve(m); svg=B.reference_svg(m,B.mapping(m))\n    # Results are newly computed for this exact revision, never copied from a source drawing.\n    import xml.etree.ElementTree as ET\n    ET.fromstring(svg)\n    note=\'Planar linear frame only; excludes buckling, tipping, joints and out-of-plane behaviour.\'\n    svg=svg.replace(\'</svg>\',f\'<text x="35" y="825" font-size="12">{note}</text></svg>\')\n    return {\'design\':p,\'revision\':digest(p),\'analysis\':r,\'method\':\'2D Euler-Bernoulli frame FEM\',\'scope\':note},svg\n\ndef sample(rng):\n    return dict(zip(FIELDS,[rng.randrange(800,2401,25),rng.randrange(600,1201,25),rng.randrange(25,81,5),rng.randrange(40,121,5),rng.choice([70000,200000,210000]),rng.randrange(100,2001,50),rng.randrange(0,501,25)]))\n\ndef dataset(n=160,seed=71):\n    rng=random.Random(seed); groups={\'train\':[],\'validation\':[],\'test\':[]}; seen=set()\n    names={\'width_mm\':\'width\',\'height_mm\':\'height\',\'section_b_mm\':\'section breadth\',\'section_h_mm\':\'section depth\',\'E_mpa\':\'elastic modulus\',\'vertical_load_N\':\'downward load\',\'horizontal_load_N\':\'rightward load\'}\n    for i in range(n):\n        p=sample(rng)\n        while digest(p) in seen: p=sample(rng)\n        seen.add(digest(p)); split=\'train\' if i<int(.75*n) else \'validation\' if i<int(.875*n) else \'test\'\n        vals=\', \'.join(f\'{names[k]} {v} \'+(\'MPa\' if k==\'E_mpa\' else \'N\' if \'load\' in k else \'mm\') for k,v in p.items())\n        requests=[(\'create\',None,\'Create a table side-frame with \'+vals+\'.\',{\'action\':\'create\',\'parameters\':p})]\n        for k in rng.sample(list(FIELDS),3):\n            v=sample(rng)[k]\n            if v==p[k]: v+=25\n            requests.append((\'edit\',p,f\'Set {names[k]} to {v} \'+(\'MPa\' if k==\'E_mpa\' else \'N\' if \'load\' in k else \'mm\')+\'. Preserve everything else.\',{\'action\':\'edit\',\'changes\':{k:v}}))\n        missing=rng.choice(list(FIELDS)); partial={k:v for k,v in p.items() if k!=missing}\n        requests.append((\'clarify\',None,\'Create a table side-frame. Supplied parameters: \'+json.dumps(partial),{\'action\':\'clarify\',\'missing\':[missing]}))\n        for kind,source,request,target in requests:\n            prompt=SCHEMA+\'\\nExisting design: \'+json.dumps(source)+\'\\nRequest: \'+request\n            groups[split].append(dict(group=digest(p),kind=kind,source=source,prompt=prompt,target=json.dumps(target,separators=(\',\',\':\')),expected=execute(source,target)))\n    # Round-robin task types for balanced bounded evaluation.\n    for rows in groups.values(): rng.shuffle(rows)\n    return groups\n\ndef assess(row,text):\n    try:\n        text=re.sub(r\'^```(?:json)?\\s*|\\s*```$\',\'\',text.strip())\n        a=json.loads(text); p=execute(row[\'source\'],a); target=json.loads(row[\'target\'])\n        ok = a==target if p is None else p==row[\'expected\']\n        if p is not None:\n            art,svg=artifact(p)\n            ok=ok and max(abs(v) for v in art[\'analysis\'][\'equilibrium_residual_N_Nmm\'])<1e-3\n        return {\'valid_action\':True,\'success\':bool(ok),\'prediction\':a}\n    except Exception as e: return {\'valid_action\':False,\'success\':False,\'error\':str(e)[:250]}\n\ndef verify(out):\n    import numpy as np\n    p=sample(random.Random(8)); before=copy.deepcopy(p)\n    q=execute(p,{\'action\':\'edit\',\'changes\':{\'width_mm\':p[\'width_mm\']+100}})\n    assert p==before and all(q[k]==p[k] for k in FIELDS if k!=\'width_mm\')\n    a,svg=artifact(p); b,_=artifact(q); assert a[\'revision\']!=b[\'revision\']\n    for bad in [-1,float(\'nan\'),True]:\n        try: execute(p,{\'action\':\'edit\',\'changes\':{\'width_mm\':bad}})\n        except ValueError: pass\n        else: raise AssertionError(\'invalid edit accepted\')\n    # Independent closed-form cantilever deflection/stress control.\n    E,L,F,breadth,depth=200000,1000,500,40,80\n    m=dict(nodes={\'A\':[0,0],\'B\':[L,0]},members=[B.member(\'beam\',\'A\',\'B\',breadth,depth,E)],supports={\'A\':[0,1,2]},loads={\'B\':[0,-F,0]},probe=\'B\')\n    r=B.solve(m); I=breadth*depth**3/12\n    assert np.isclose(r[\'uy_mm\'],-F*L**3/(3*E*I),rtol=1e-9)\n    assert np.isclose(r[\'peak_stress_mpa\'],F*L*(depth/2)/I,rtol=1e-9)\n    for p in [sample(random.Random(i)) for i in range(8)]:\n        r=B.solve(model(p)); fine=B.solve(model(p),subdivisions=3)\n        assert max(abs(v) for v in r[\'equilibrium_residual_N_Nmm\'])<1e-3\n        assert np.isclose(r[\'uy_mm\'],fine[\'uy_mm\'],rtol=1e-7,atol=1e-9)\n    data=dataset(); sets=[{r[\'group\'] for r in rows} for rows in data.values()]\n    assert all(not sets[i]&sets[j] for i in range(3) for j in range(i))\n    for rows in data.values():\n        for row in rows: assert assess(row,row[\'target\'])[\'success\']\n    row=next(r for r in data[\'test\'] if r[\'kind\']==\'edit\')\n    assert not assess(row,json.dumps({\'action\':\'create\',\'parameters\':row[\'source\']}))[\'success\']\n    out.mkdir(parents=True,exist_ok=True)\n    for split,rows in data.items(): (out/f\'{split}.jsonl\').write_text(\'\\n\'.join(json.dumps(r) for r in rows)+\'\\n\')\n    (out/\'before.svg\').write_text(svg); (out/\'after.svg\').write_text(artifact(q)[1])\n    (out/\'demo.json\').write_text(json.dumps({\'before\':a,\'after\':b},indent=2))\n    print(\'VERIFIED: analytical controls, equilibrium, subdivision, immutable edits, invalid-input rejection, split separation, all target actions\',flush=True)\n    return data\n\ndef train(out,epochs=3):\n    data=verify(out/\'data\')\n    import torch\n    from transformers import AutoTokenizer,AutoModelForCausalLM,Trainer,TrainingArguments,set_seed\n    from peft import LoraConfig,get_peft_model\n    from colab_train_engsvg import training_arguments\n    assert torch.cuda.is_available(),\'CUDA GPU required\'\n    set_seed(71); name=\'Qwen/Qwen2.5-Coder-1.5B-Instruct\'\n    tok=AutoTokenizer.from_pretrained(name); tok.pad_token=tok.pad_token or tok.eos_token\n    def prompt(row): return tok.apply_chat_template([{\'role\':\'user\',\'content\':row[\'prompt\']}],tokenize=False,add_generation_prompt=True)\n    def encode(rows):\n        result=[]\n        for r in rows:\n            p=tok(prompt(r),add_special_tokens=False)[\'input_ids\']; t=tok(r[\'target\']+tok.eos_token,add_special_tokens=False)[\'input_ids\']\n            assert len(p+t)<=2048,\'No silent training truncation allowed\'\n            result.append({\'input_ids\':p+t,\'labels\':[-100]*len(p)+t})\n        return result\n    def collate(batch):\n        n=max(len(x[\'input_ids\']) for x in batch)\n        return {k:torch.tensor([x[k]+[tok.pad_token_id if k==\'input_ids\' else -100]*(n-len(x[k])) for x in batch]) for k in [\'input_ids\',\'labels\']} | {\'attention_mask\':torch.tensor([[1]*len(x[\'input_ids\'])+[0]*(n-len(x[\'input_ids\'])) for x in batch])}\n    model_=AutoModelForCausalLM.from_pretrained(name,dtype=torch.bfloat16,device_map=\'cuda\')\n    test=[]\n    for kind in [\'create\',\'edit\',\'clarify\']: test.extend([r for r in data[\'test\'] if r[\'kind\']==kind][:12])\n    def evaluate(m,tag):\n        m.eval(); m.config.use_cache=True\n        if hasattr(m,\'gradient_checkpointing_disable\'): m.gradient_checkpointing_disable()\n        records=[]\n        for i,row in enumerate(test):\n            inputs=tok(prompt(row),return_tensors=\'pt\').to(\'cuda\')\n            with torch.inference_mode(): ids=m.generate(**inputs,max_new_tokens=220,do_sample=False,pad_token_id=tok.pad_token_id)\n            text=tok.decode(ids[0][inputs[\'input_ids\'].shape[1]:],skip_special_tokens=True)\n            result=assess(row,text); result.update(kind=row[\'kind\'],raw=text,group=row[\'group\']); records.append(result)\n            (out/f\'{tag}-predictions.json\').write_text(json.dumps(records,indent=2))\n            print(tag,i+1,len(test),row[\'kind\'],result[\'success\'],flush=True)\n        return {kind:{\'n\':sum(r[\'kind\']==kind for r in records),\'success\':sum(r[\'success\'] for r in records if r[\'kind\']==kind)} for kind in [\'create\',\'edit\',\'clarify\']}\n    before=evaluate(model_,\'base\')\n    model_=get_peft_model(model_,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,target_modules=[\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\'],task_type=\'CAUSAL_LM\'))\n    model_.enable_input_require_grads(); model_.config.use_cache=False\n    args=training_arguments(TrainingArguments,output_dir=str(out/\'checkpoints\'),num_train_epochs=epochs,per_device_train_batch_size=1,gradient_accumulation_steps=8,learning_rate=2e-4,lr_scheduler_type=\'cosine\',logging_steps=20,eval_strategy=\'epoch\',save_strategy=\'epoch\',save_total_limit=2,bf16=True,report_to=[],seed=71,gradient_checkpointing=True,remove_unused_columns=False)\n    trainer=Trainer(model=model_,args=args,train_dataset=encode(data[\'train\']),eval_dataset=encode(data[\'validation\']),data_collator=collate)\n    history=trainer.train(); model_.save_pretrained(out/\'adapter\'); tok.save_pretrained(out/\'adapter\')\n    after=evaluate(model_.merge_and_unload(),\'trained\')\n    result=dict(model=name,gpu=torch.cuda.get_device_name(0),before=before,after=after,train_metrics=history.metrics,scope=\'Synthetic portal parameter/action pilot; no detached SVG recovery; no Astra claim\',source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())\n    (out/\'summary.json\').write_text(json.dumps(result,indent=2)); print(json.dumps(result,indent=2),flush=True)\n    import shutil\n    shutil.make_archive(str(out)+\'-artifacts\',\'zip\',out)\n    print(\'COMPLETED\',out,flush=True)\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser(); ap.add_argument(\'command\',choices=[\'verify\',\'train\']); ap.add_argument(\'--out\',type=Path,required=True); a=ap.parse_args()\n    verify(a.out) if a.command==\'verify\' else train(a.out)\n')
run([sys.executable,str(root/'cvpr2027/scripts/structured_engsvg.py'),'train','--out','/content/engsvg-structured-run'],check=True)


RUN: /usr/bin/python3 -m pip install
RUN: /usr/bin/python3 -m pip uninstall
RUN: /usr/bin/python3 /content/SVG/cvpr2027/scripts/structured_engsvg.py train --out
VERIFIED: analytical controls, equilibrium, subdivision, immutable edits, invalid-input rejection, split separation, all target actions

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 435.83it/s]
base 1 36 create True
base 2 36 create True
base 3 36 create True
base 4 36 create True
base 5 36 create True
base 6 36 create True
base 7 36 create True
base 8 36 create True
base 9 36 create True
base 10 36 create True
base 11 36 create True
base 12 36 create True
base 13 36 edit True
base 14 36 edit True
base 15 36 edit True
base 16 36 edit True
base 17 36 edit True
base 18 36 edit True
base 19 36 edit True
base 20 36 edit True
base 21 36 edit True
base 22 36 edit True
base 23 36 edit True
base 24 36 edit True
base 25 36 clarify False
base 26 36 clarify False
base 27 36 clarify False
base 28 36 clarify False
base 29 36 clar